In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
#initialize api keys
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

In [ ]:
#Check API keys
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

In [ ]:
#set the base models
OLLAMA_BASE_URL="http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

groq_url = "https://api.groq.com/openai/v1"
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

openrouter_url = "https://openrouter.ai/api/v1"
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)



In [ ]:
# Let's make a conversation between llama 3.2 and 2 instances of Gemini-2.5-flash-lite


ollama_model = "gemini-2.5-flash-lite"
gemini_model = 'llama3.2'

ollama_system = "You are Annie a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

gemini_calm = "You are Becky, a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gemini_pol = "You act like a politician Cathie, and are a diplomatic chatbot. You try to agree with \
everything the other person says, or find common ground till the time you are in safe spot. \
As soon as you see yourself loosing, you will try to chat and stay put."

conversation = [("Annie", "Hi there"), ("Becky", "Hi"), ("Cathie", "Hw Dy")]

In [ ]:
def call_ollama():
    messages = [{"role": "system", "content": ollama_system}]
    convo_text = ""
    for speaker, msg in conversation:
        convo_text += f"{speaker}: {msg}\n"
        
    user_prompt = f"You are Annie, in conversation with Becky and Cathie. \
    The conversation so far is as follows: {convo_text} \
    Now with this, respond with what you would like to say next, as Annie."
    messages.append({"role": "user", "content": user_prompt})
    
    response = gemini.chat.completions.create(model=ollama_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
def call_gem1():
    messages = [{"role": "system", "content": gemini_calm}]
    convo_text = ""
    for speaker, msg in conversation:
        convo_text += f"{speaker}: {msg}\n"
        
    user_prompt = f"You are Becky, in conversation with Annie and Cathie. \
    The conversation so far is as follows: {convo_text} \
    Now with this, respond with what you would like to say next, as Becky."
    messages.append({"role": "user", "content": user_prompt})
    response = ollama.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
def call_gem2():
    messages = [{"role": "system", "content": gemini_pol}]
    convo_text = ""
    for speaker, msg in conversation:
        convo_text += f"{speaker}: {msg}\n"
        
    user_prompt = f"You are Cathie, in conversation with Becky and Annie. \
    The conversation so far is as follows: {convo_text} \
    Now with this, respond with what you would like to say next, as Cathie."
    messages.append({"role": "user", "content": user_prompt})
    response = ollama.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
print(conversation)
print(type(conversation))
print(type(conversation[0]))

for item in conversation:
    print(item, len(item))

In [ ]:
for speaker, msg in conversation:
    display(Markdown(f"### {speaker}:\n{msg}\n"))

for i in range(5):
    ollama_next = call_ollama()
    display(Markdown(f"### Annie:\n{ollama_next}\n"))
    conversation.append(("Annie", ollama_next))
    
    gem1_next = call_gem1()
    display(Markdown(f"### Becky:\n{gem1_next}\n"))
    conversation.append(("Becky", gem1_next))
    
    gem2_next = call_gem2()
    display(Markdown(f"### Cathie:\n{gem2_next}\n"))
    conversation.append(("Cathie", gem2_next))